# TP53 Gene Mutation Prediction from METABRIC Breast Cancer RNA Expression
## AI3013 Machine Learning Course Project

**All models implemented from scratch using only NumPy, Pandas, and Matplotlib.**

## 1. Environment & Imports

In [ ]:
import numpy as np
from IPython.display import Image, display
import sys
sys.path.insert(0, '.')

## 2. Load Data

In [ ]:
from SourceCode.data_loader import load_tp53_data

X, y = load_tp53_data('dataset/METABRIC_RNA_Mutation.csv')
print(f'Shape: {X.shape}  |  TP53 mutated: {np.sum(y==1)}  |  Wild-type: {np.sum(y==0)}')

## 3. Train/Test Split & Standardization

80% training / 20% test. Z-score standardization: $(x - \mu) / \sigma$.
Test set uses training mean & std to prevent data leakage.

In [ ]:
from SourceCode.preprocessing import custom_train_test_split, custom_standard_scaler

X_train, X_test, y_train, y_test = custom_train_test_split(X, y, test_size=0.2)
X_train, X_test = custom_standard_scaler(X_train, X_test)

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'Train pos/neg: {np.sum(y_train==1)}/{np.sum(y_train==0)}')

## 4. Run All Experiments

5 experiments + regularization paths + learning curves:

In [ ]:
from SourceCode.experiment import run_all_experiments

data = run_all_experiments(X_train, X_test, y_train, y_test)

## 5. Hold-out Test Results

In [18]:
results = data['results']
X_n_features = X_train.shape[1]

print(f"{'Model':30s}  {'Acc':>7s}  {'F1':>7s}  {'Prec':>7s}  {'Rec':>7s}  {'Features':>12s}")
print('-' * 85)
for name, (acc, f1, prec, rec, _, feat) in results.items():
    feat_str = f'{feat}d' if isinstance(feat, int) else str(feat)
    print(f'{name:30s}  {acc:7.4f}  {f1:7.4f}  {prec:7.4f}  {rec:7.4f}  {feat_str:>12s}')

majority = max(np.mean(y_test == 0), np.mean(y_test == 1))
print(f"{'Majority baseline':30s}  {majority:7.4f}  {'-':>7s}  {'-':>7s}  {'-':>7s}  {'-':>12s}")

Model                               Acc       F1     Prec      Rec      Features
-------------------------------------------------------------------------------------
LR + L2                          0.8526   0.7895   0.8015   0.7778          489d
LR + L1                          0.8868   0.8300   0.8898   0.7778          106d
PCA + L2                         0.8605   0.7954   0.8306   0.7630          312d
SVM Linear                       0.8579   0.8099   0.7718   0.8519          489d
SVM RBF                          0.9000   0.8582   0.8647   0.8519      g=0.0010
Majority baseline                0.6447        -        -        -             -


## 6. 5-Fold Cross-Validation Results (mean ± std)

In [19]:
cv_results = data['cv_results']

print(f"{'Model':30s}  {'Acc':>15s}  {'F1':>15s}  {'Prec':>15s}  {'Rec':>15s}")
print('-' * 85)
for name, cv in cv_results.items():
    print(f"{name:30s}  {cv['accuracy'][0]:.4f}±{cv['accuracy'][1]:.4f}  "
          f"{cv['f1'][0]:.4f}±{cv['f1'][1]:.4f}  "
          f"{cv['precision'][0]:.4f}±{cv['precision'][1]:.4f}  "
          f"{cv['recall'][0]:.4f}±{cv['recall'][1]:.4f}")

Model                                       Acc               F1             Prec              Rec
-------------------------------------------------------------------------------------
LR + L2                         0.8150±0.0196  0.7301±0.0201  0.7328±0.0217  0.7281±0.0286
LR + L1                         0.8504±0.0203  0.7671±0.0350  0.8212±0.0353  0.7207±0.0431
PCA + L2                        0.8077±0.0133  0.7223±0.0202  0.7160±0.0281  0.7294±0.0211
SVM Linear                      0.8130±0.0242  0.7453±0.0276  0.7013±0.0339  0.7959±0.0247
SVM RBF                         0.8550±0.0186  0.7979±0.0275  0.7637±0.0180  0.8362±0.0449


## 7. Key Findings

In [20]:
svm_rbf_acc = results['SVM RBF'][0]
lr_l1_acc = results['LR + L1'][0]
svm_lin_acc = results['SVM Linear'][0]
l1_indices = data['l1_indices']

print(f'SVM RBF  ({svm_rbf_acc:.4f}) > SVM Linear ({svm_lin_acc:.4f})  ' +
      f'-> Delta = +{(svm_rbf_acc - svm_lin_acc)*100:.1f}%  (nonlinear structure detected)')
print(f'SVM RBF  ({svm_rbf_acc:.4f}) > LR + L1    ({lr_l1_acc:.4f})  ' +
      f'-> Delta = +{(svm_rbf_acc - lr_l1_acc)*100:.1f}%  (nonlinear > sparse linear)')
print(f'LR + L1 uses only {len(l1_indices)}/{X_n_features} features but achieves {lr_l1_acc:.4f}  ' +
      '(interpretability vs. accuracy trade-off)')

SVM RBF  (0.9000) > SVM Linear (0.8579)  -> Delta = +4.2%  (nonlinear structure detected)
SVM RBF  (0.9000) > LR + L1    (0.8868)  -> Delta = +1.3%  (nonlinear > sparse linear)
LR + L1 uses only 106/489 features but achieves 0.8868  (interpretability vs. accuracy trade-off)


## 8. Visualizations

### 8.1 Model Performance Comparison (Hold-out + 5-Fold CV)

In [ ]:
from SourceCode import visualization as viz
viz.plot_metrics_comparison(results, cv_results)
display(Image('plots/01_metrics_comparison.png'))

### 8.2 Cost Convergence Curves (LR + SVM)

In [ ]:
viz.plot_cost_curves(data['cost_data'])
display(Image('plots/02_cost_curves.png'))

### 8.3 PCA Cumulative Explained Variance

Horizontal dashed line = 95% threshold (312 components selected from 489).

In [ ]:
viz.plot_pca_variance(data['pca'].explained_variance_ratio_)
display(Image('plots/03_pca_variance.png'))

### 8.4 L1 Regularization — Top 30 Selected Genes

Blue = positive association with mutation; Red = negative association. Only 106/489 genes retained.

In [ ]:
viz.plot_l1_feature_weights(data['l1_indices'], data['l1_weights'], X_n_features)
display(Image('plots/04_l1_feature_weights.png'))

### 8.5 Regularization Paths — CV Accuracy vs. Lambda

In [ ]:
for penalty, (lambda_values, cv_scores) in data['reg_path_data'].items():
    viz.plot_regularization_path(lambda_values, cv_scores, penalty.upper())
    display(Image(f'plots/regularization_path_{penalty.upper()}.png'))

### 8.6 Learning Curves — Overfitting & Underfitting Analysis

Gap between training (blue) and CV validation (red) indicates overfitting.

In [ ]:
for name, (train_sizes, train_scores, val_scores) in data['lc_data'].items():
    viz.plot_learning_curve(train_sizes, train_scores, val_scores, name)
    display(Image(f'plots/learning_curve_{name.replace(" ", "_").replace("+", "")}.png'))

## 9. Model Summary

| Model | Key Feature | Best For |
|---|---|---|
| LR + L2 (Ridge) | All features, small weights | Baseline linear model |
| LR + L1 (Lasso) | **106/489 features** selected | Interpretability, gene discovery |
| PCA + LR + L2 | 312 PCs (95% variance) | Dimensionality reduction study |
| SVM Linear | Max-margin, class-balanced | Robust linear classifier |
| SVM RBF | **Best accuracy**, nonlinear kernel | Captures nonlinear decision boundaries |

---
*AI3013 Machine Learning Group Project — TP53 Mutation Prediction*